# 02 — Preprocessing: Multicollinearity Check (VIF)

**Purpose**: this notebook's first and primary step is a Variance Inflation
Factor (VIF) check across all 20 input features, directly following up on
the EDA finding (`01_EDA.ipynb`) that mean |correlation| across the 210
feature pairs was 0.58, with 17/210 pairs exceeding |r| = 0.7.

Per `IMPLEMENTATION_RULES.md`, multicollinearity must be checked and
reported before any model is considered ready. **This notebook only
computes and reports VIF — no features are dropped, transformed, encoded,
or scaled here.** Those decisions are made explicitly, afterward, once the
numbers have been reviewed.

Research World only (`ml_pipeline/`) — no artifacts are produced by this
notebook.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

DATA_PATH = "../datasets/raw/student_stress_factors.csv"
STRESS_LABEL_COL = "stress_level"
VIF_THRESHOLD = 10.0

## Load dataset and isolate input features

VIF is a property of the *input feature set*, not the target — `stress_level`
is excluded before computing it.

In [2]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=[STRESS_LABEL_COL])
print(f"Input feature matrix: {X.shape[0]} rows x {X.shape[1]} features")
list(X.columns)

Input feature matrix: 1100 rows x 20 features


['anxiety_level',
 'self_esteem',
 'mental_health_history',
 'depression',
 'headache',
 'blood_pressure',
 'sleep_quality',
 'breathing_problem',
 'noise_level',
 'living_conditions',
 'safety',
 'basic_needs',
 'academic_performance',
 'study_load',
 'teacher_student_relationship',
 'future_career_concerns',
 'social_support',
 'peer_pressure',
 'extracurricular_activities',
 'bullying']

## Compute VIF

No `statsmodels` dependency in `mainks` — VIF is implemented directly with
`scikit-learn`, which is already installed, using the standard definition:
for each feature $X_i$, fit an OLS regression of $X_i$ on all other
features, take its $R^2_i$, and compute

$$\text{VIF}_i = \frac{1}{1 - R^2_i}$$

This is mathematically identical to `statsmodels.stats.outliers_influence.
variance_inflation_factor` when the regression includes an intercept (which
`sklearn.linear_model.LinearRegression` does by default).

In [3]:
def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    """Compute Variance Inflation Factor for every column of X."""
    vif_rows = []
    for col in X.columns:
        y_target = X[col]
        X_others = X.drop(columns=[col])
        r2 = LinearRegression().fit(X_others, y_target).score(X_others, y_target)
        vif = np.inf if r2 >= 1.0 else 1.0 / (1.0 - r2)
        vif_rows.append({"feature": col, "r_squared": r2, "VIF": vif})
    return pd.DataFrame(vif_rows).sort_values("VIF", ascending=False).reset_index(drop=True)

vif_table = compute_vif(X)
vif_table

,feature,r_squared,VIF
0,social_support,0.825959,5.745766
1,blood_pressure,0.728752,3.686664
2,future_career_concerns,0.707310,3.416580
3,anxiety_level,0.690055,3.226378
4,self_esteem,0.688315,3.208365
5,teacher_student_relationship,0.687331,3.198274
6,bullying,0.686266,3.187411
7,depression,0.676428,3.090505
8,sleep_quality,0.676279,3.089081
9,safety,0.641595,2.790136


## Features exceeding the VIF < 10 threshold

In [4]:
over_threshold = vif_table[vif_table["VIF"] >= VIF_THRESHOLD].copy()
over_threshold["excess_over_threshold"] = over_threshold["VIF"] - VIF_THRESHOLD

print(f"Threshold: VIF < {VIF_THRESHOLD}")
print(f"Features exceeding threshold: {len(over_threshold)} / {len(vif_table)}")
print()
if len(over_threshold) > 0:
    print(over_threshold.to_string(index=False))
else:
    print("None — all features are under the VIF < 10 threshold.")

Threshold: VIF < 10.0
Features exceeding threshold: 0 / 20

None — all features are under the VIF < 10 threshold.


---

# Systematic Target-Leakage Audit

**Added after `04_SHAPAnalysis.ipynb` revealed that `blood_pressure` alone
reproduces the full model's accuracy.** This section asks the obvious next
question: is `blood_pressure` uniquely broken, or is it just the worst
example of a dataset-wide problem?

This audit is deliberately placed here, in the preprocessing/data-quality
notebook, but was written *after* the SHAP work — the chronology matters for
the methodology write-up and is preserved rather than tidied away.

Method — three complementary tests, because no single one is conclusive:

1. **Purity scan.** For every value of every feature, what share of rows fall
   into a single class? High purity alone is *not* proof of leakage — a
   genuinely strong ordinal predictor produces high-purity buckets too.
2. **Single-feature lookup test.** Fit a trivial "most common class per
   value" rule on the training split only, score it on the same held-out
   test set. This is the test that exposed `blood_pressure`; it directly
   answers "how much of the model's performance does this one feature
   explain?"
3. **Monotonicity.** A near-deterministic *and* non-monotonic mapping
   (value order not matching class order) is the strongest single indicator
   of a synthetically-generated column, because it is implausible for a real
   ordinal measurement while being invisible to Pearson correlation.

In [5]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2

y = df[STRESS_LABEL_COL]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

audit_rows = []
for feat in X.columns:
    # (2) single-feature lookup rule, learned on train only
    rule = X_tr.assign(_y=y_tr).groupby(feat)["_y"].agg(lambda s: s.mode()[0])
    pred = X_te[feat].map(rule).fillna(y_tr.mode()[0])
    lookup_acc = accuracy_score(y_te, pred)

    norm = pd.crosstab(df[feat], df[STRESS_LABEL_COL], normalize="index")
    counts = df[feat].value_counts()

    # (1) purity
    perfect_vals = norm.index[norm.max(axis=1) == 1.0]
    pct_perfect = counts[perfect_vals].sum() / len(df) if len(perfect_vals) else 0.0
    max_purity = norm.max(axis=1).max()

    # (3) monotonicity of dominant class across ascending feature values
    dom = norm.idxmax(axis=1).values
    monotonic = bool(np.all(np.diff(dom) >= 0) or np.all(np.diff(dom) <= 0))

    audit_rows.append(
        {
            "feature": feat,
            "lookup_acc": lookup_acc,
            "max_purity": max_purity,
            "pct_perfectly_determined": pct_perfect,
            "dominant_class_monotonic": monotonic,
            "n_unique": df[feat].nunique(),
        }
    )

audit = (
    pd.DataFrame(audit_rows).sort_values("lookup_acc", ascending=False).reset_index(drop=True)
)
audit.index = audit.index + 1

FULL_MODEL_ACC = 0.8864  # tuned 20-feature Random Forest, from 03_ModelTraining.ipynb
print(f"Reference: tuned 20-feature Random Forest accuracy = {FULL_MODEL_ACC}\n")
print(audit.round(4).to_string())

Reference: tuned 20-feature Random Forest accuracy = 0.8864

                         feature  lookup_acc  max_purity  pct_perfectly_determined  dominant_class_monotonic  n_unique
1                  sleep_quality      0.9045      0.9573                    0.0000                     False         6
2         future_career_concerns      0.9000      0.9572                    0.0000                     False         6
3                 blood_pressure      0.8864      1.0000                    0.5455                     False         3
4                     depression      0.8818      1.0000                    0.0191                      True        28
5                       bullying      0.8818      0.9313                    0.0000                      True         6
6                  anxiety_level      0.8773      0.9524                    0.0000                      True        22
7                       headache      0.8182      0.9426                    0.0000                      Tr

In [6]:
# How many single features individually rival or beat the full 20-feature model?
rivals = audit[audit["lookup_acc"] >= FULL_MODEL_ACC - 0.02]
print(f"Features whose ONE-value lookup rule scores within 0.02 of the full model "
      f"({FULL_MODEL_ACC}):\n")
print(rivals[["feature", "lookup_acc", "dominant_class_monotonic"]].round(4).to_string())
print(f"\n{len(rivals)} of {len(audit)} features individually rival the entire tuned model.")
beats = audit[audit["lookup_acc"] > FULL_MODEL_ACC]
print(f"{len(beats)} features individually BEAT it: {list(beats['feature'])}")

Features whose ONE-value lookup rule scores within 0.02 of the full model (0.8864):

                  feature  lookup_acc  dominant_class_monotonic
1           sleep_quality      0.9045                     False
2  future_career_concerns      0.9000                     False
3          blood_pressure      0.8864                     False
4              depression      0.8818                      True
5                bullying      0.8818                      True
6           anxiety_level      0.8773                      True

6 of 20 features individually rival the entire tuned model.
2 features individually BEAT it: ['sleep_quality', 'future_career_concerns']


## Result: the leakage is dataset-wide, not localised to `blood_pressure`

`sleep_quality` (0.9045) and `future_career_concerns` (0.9000) each **beat**
the entire tuned 20-feature Random Forest (0.8864) on their own, from a
single ordinal column. `blood_pressure` (0.8864) merely ties it. Six features
land within 0.01 of the full model.

A single self-reported 0–5 item cannot genuinely predict another
self-reported item at 90% accuracy across three classes. Real psychological
survey data is far noisier than this.

The crosstabs below show the shared generative signature.

In [7]:
for feat in ["sleep_quality", "future_career_concerns", "blood_pressure", "social_support"]:
    ct = pd.crosstab(df[feat], df[STRESS_LABEL_COL], normalize="index").round(3)
    ct["n_rows"] = df[feat].value_counts().sort_index()
    print("=" * 60)
    print(feat)
    print("=" * 60)
    print(ct)
    print()

sleep_quality
stress_level       0      1      2  n_rows
sleep_quality                             
0              0.382  0.265  0.353      34
1              0.024  0.018  0.957     328
2              0.069  0.904  0.027     188
3              0.082  0.847  0.071     170
4              0.862  0.090  0.048     188
5              0.849  0.062  0.089     192

future_career_concerns
stress_level                0      1      2  n_rows
future_career_concerns                             
0                       0.233  0.300  0.467      30
1                       0.957  0.009  0.034     327
2                       0.075  0.864  0.060     199
3                       0.058  0.873  0.069     173
4                       0.081  0.065  0.854     185
5                       0.070  0.059  0.871     186

blood_pressure
stress_level        0      1      2  n_rows
blood_pressure                             
1               0.000  1.000  0.000     300
2               1.000  0.000  0.000     300
3         

Every one of these shows the same structure:

- a **small, near-uniform bucket at value 0** (n = 30–88, split roughly
  evenly across all three classes) — behaving like a missing-data sentinel;
- **every other value mapping to one class at 85–100% purity.**

`sleep_quality = 1` → high stress in 95.7% of rows. `future_career_concerns
= 1` → low stress in 95.7%. `social_support = 2` → moderate stress in
**100%** of 142 rows. `teacher_student_relationship = 5` → low stress in
**100%** of 140 rows.

This is the signature of **features sampled conditional on the label** during
synthetic data generation — not of measured human responses.

The final test: if leakage were localised, removing the worst offenders
should collapse performance toward chance. It does not.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

lookup_map = dict(zip(audit["feature"], audit["lookup_acc"]))

scenarios = [
    ("ALL 20 features (original)", list(X.columns)),
    ("drop blood_pressure only", [c for c in X.columns if c != "blood_pressure"]),
    ("drop lookup_acc > 0.85", [c for c in X.columns if lookup_map[c] <= 0.85]),
    ("drop lookup_acc > 0.80", [c for c in X.columns if lookup_map[c] <= 0.80]),
    ("drop lookup_acc > 0.70 (6 weakest kept)", [c for c in X.columns if lookup_map[c] <= 0.70]),
]

print(f"{'scenario':44s} {'n_feat':>6s} {'accuracy':>9s} {'f1_macro':>9s}")
print("-" * 72)
removal_results = []
for label, cols in scenarios:
    m = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
    m.fit(X_tr[cols], y_tr)
    p = m.predict(X_te[cols])
    a, f = accuracy_score(y_te, p), f1_score(y_te, p, average="macro")
    removal_results.append({"scenario": label, "n_features": len(cols), "accuracy": a, "f1_macro": f})
    print(f"{label:44s} {len(cols):6d} {a:9.4f} {f:9.4f}")

print("\nChance level for 3 balanced classes = 0.3333")

scenario                                     n_feat  accuracy  f1_macro
------------------------------------------------------------------------


ALL 20 features (original)                       20    0.8864    0.8861
drop blood_pressure only                         19    0.8773    0.8768


drop lookup_acc > 0.85                           14    0.8864    0.8865
drop lookup_acc > 0.80                           12    0.8727    0.8728


drop lookup_acc > 0.70 (6 weakest kept)           6    0.8500    0.8498

Chance level for 3 balanced classes = 0.3333


### Conclusion of the audit

Removing leaky features does not recover an honest signal, because there is
no unleaked signal underneath to recover:

- dropping `blood_pressure` costs **0.009** accuracy;
- dropping the three worst offenders costs **nothing at all** (0.8864 →
  0.8864) — the remaining features carry the same label information;
- keeping only the **six weakest** features still yields **0.85**, against a
  0.333 chance baseline.

**Feature removal is not a viable remedy for this dataset.** The label
information is redundantly encoded across essentially every column, so
excluding any subset simply shifts the model onto the next proxy.

**Implications, carried into `methodology.md` and `model_card.md`:**

1. No accuracy figure computed on this dataset — original or corrected —
   can be presented as evidence of real-world stress-prediction capability.
2. The corrected comparison run in `03_ModelTraining.ipynb` (with
   `blood_pressure` excluded) is still worth reporting, but as *evidence
   that removal does not work*, not as a fixed result.
3. The Phase 8 externally-collected validation set moves from a planned
   strengthening step to **the only route by which this project can make any
   empirical claim about real predictive performance.**
4. The framework, pipeline, and explainability method remain valid and are
   what surfaced this. That is a defensible research contribution
   independent of the benchmark dataset's quality.